# RAG 评估指标 · 03 — Answer Relevancy & MRR

**本 Notebook 目标：** 演示答案相关性评分（逆向问题生成 + 语义相似度）和 MRR@k 的计算方法。

## 学习目标

- 理解 Answer Relevancy：答案是否真正回应了用户问题
- 掌握逆向问题生成（Reverse Question Generation）评估思路
- 计算 MRR / MRR@k，评估检索器排序质量

```bash
pip install openai python-dotenv
```

## 本 Notebook 两大模块

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    subgraph AR[Answer Relevancy 生成侧]
        Q[Question] --> ANS[Answer]
        ANS --> REV[逆向生成 Q prime]
        Q --> SIM[语义相似度]
        REV --> SIM
        SIM --> ARS[Relevancy 得分]
    end
    subgraph MRR[MRR 检索侧]
        RET[retrieved 列表] --> RR[Reciprocal Rank]
        RR --> MR[MRR 均值]
    end
```

---
# Part 1 · 环境准备

从上级目录 `04_RAG_Evaluation/.env` 加载 `OPENAI_API_KEY`。

In [ ]:
import os
import json
import math
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

api_key = "sk-your-api-key-here"


client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com",
)
print("✅ DeepSeek 客户端已初始化")

---
# Chapter 5 · Answer Relevancy 深入

> 你的回答，是在回答用户真正想问的问题吗？

## Answer Relevancy 直觉

**为什么不直接算正确性（Correctness）？**

| 方法 | 问题 |
|------|------|
| 字符串匹配 | 同义表达无法识别，"机器学习"≠"ML" |
| 标准答案对比 | 需要完美标准答案，标注成本高 |
| LLM 直接打分 | 主观性强，不可复现 |

**关键假设**：一个好的答案，应该能让 LLM 准确"猜"到原始问题是什么。

## 反向验证算法

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    ANS[Answer A] --> LLM[LLM 逆向生成]
    LLM --> Q1[Q prime 1]
    LLM --> Q2[Q prime 2]
    LLM --> Q3[Q prime 3]
    Q[原始 Question Q] --> COS[cos 相似度]
    Q1 --> COS
    Q2 --> COS
    Q3 --> COS
    COS --> AVG[Answer Relevancy = 均值]
```

$$\text{Answer Relevancy} = \frac{1}{N}\sum_{i=1}^{N} \cos(\vec{Q}, \vec{Q'_i})$$

## 反向问题生成示例

**答案：**
> RAG 通过将检索到的相关文档片段作为上下文传给 LLM，使 LLM 能够基于最新知识生成回答，而不依赖训练数据。

**生成的问题：**

- Q'₁: "什么是 RAG，它是如何工作的？"
- Q'₂: "RAG 技术是如何让 LLM 使用最新知识的？"
- Q'₃: "RAG 和普通 LLM 有什么区别？"

**原始问题 Q:** "RAG 是如何工作的？"

cos(Q, Q'₁) = 0.92 → Answer Relevancy = (0.92 + 0.87 + 0.71) / 3 = **0.833**

## Cosine Similarity 计算

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    QA[文本 Q] --> E1[Embedding]
    QB[文本 Q prime] --> E2[Embedding]
    E1 --> VQ[向量 v_Q]
    E2 --> VQ2[向量 v_Q prime]
    VQ --> COS[cos = dot / norm]
    VQ2 --> COS
```

| 值域 | 含义 |
|------|------|
| 1.0 | 语义完全相同 |
| 0.8-1.0 | 高度相似 |
| 0.6-0.8 | 中等相似 |
| < 0.5 | 低相似（跑题） |

> 本 Notebook 使用 **Jaccard 字符相似度**作为 embedding 的轻量替代；配置 API Key 后可用 LLM 逆向问题生成 + 相似度评估。

---
# Part 2 · Answer Relevancy（代码实战）

## 代码实现流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    ANS[answer] --> GQ[generate_questions_from_answer]
    GQ --> QLIST[generated_questions]
    QLIST --> AR[answer_relevancy_score]
    Q[original_question] --> AR
    AR --> SC[cosine_sim_keyword 均值]
```

In [ ]:
QA_EXAMPLES = [
    {
        "label": "高相关（直接回答问题）",
        "question": "什么是 RAG 技术？",
        "answer": "RAG（检索增强生成）是一种将检索系统与大语言模型结合的技术，通过检索相关文档并注入上下文来提升回答质量。",
    },
    {
        "label": "中等相关（部分回答）",
        "question": "什么是 RAG 技术？",
        "answer": "大语言模型有很多应用场景，包括文本生成、代码补全等。检索系统在信息获取中发挥重要作用。",
    },
    {
        "label": "低相关（答非所问）",
        "question": "什么是 RAG 技术？",
        "answer": "今天天气不错，非常适合出门散步。明天可能会下雨，记得带伞。",
    },
]


def tokenize(text: str) -> set:
    """简单中文分词：以字符为单位，过滤标点和空格"""
    stop_chars = set("，。！？、；：\"\"''（）【】《》 \n\t,.!?;:()'\"[]<>")
    return set(ch for ch in text if ch not in stop_chars)


def cosine_sim_keyword(text_a: str, text_b: str) -> float:
    """基于字符集合的 Jaccard 相似度，作为余弦相似度的简易代理"""
    set_a = tokenize(text_a)
    set_b = tokenize(text_b)
    if not set_a or not set_b:
        return 0.0
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union


def generate_questions_from_answer(answer: str, llm_client: OpenAI) -> list:
    """调用 DeepSeek，基于答案内容逆向生成 3 个可能的问题"""
    prompt = (
        "你是一位擅长从文本中提炼问题的专家。"
        "请根据以下答案，逆向生成 3 个该答案可能在回答的问题。\n\n"
        f"答案：\n{answer}\n\n"
        "请以 JSON 数组格式返回，每个元素是一个问题字符串。示例：\n"
        '["问题1", "问题2", "问题3"]\n\n'
        "只返回 JSON 数组，不要包含任何其他文字。"
    )
    response = llm_client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    raw = response.choices[0].message.content.strip()
    try:
        questions = json.loads(raw)
        if isinstance(questions, list):
            return questions
    except json.JSONDecodeError:
        start = raw.find("[")
        end = raw.rfind("]") + 1
        if start != -1 and end > start:
            try:
                return json.loads(raw[start:end])
            except json.JSONDecodeError:
                pass
    return [raw]


def answer_relevancy_score(original_question: str, generated_questions: list) -> float:
    """原始问题与每个逆向生成问题的相似度均值"""
    if not generated_questions:
        return 0.0
    scores = [cosine_sim_keyword(original_question, q) for q in generated_questions]
    return sum(scores) / len(scores)

### 运行 Answer Relevancy 评估

In [ ]:
print("=" * 60)
print("第一节：Answer Relevancy（答案相关性）评估")
print("=" * 60)
print(f"\n{'标签':<18} {'相关性分数':>10}")
print("-" * 40)

for ex in QA_EXAMPLES:
    if client:
        gen_questions = generate_questions_from_answer(ex["answer"], client)
        score = answer_relevancy_score(ex["question"], gen_questions)
        method = "LLM逆向生成"
    else:
        score = cosine_sim_keyword(ex["question"], ex["answer"])
        gen_questions = ["（LLM 不可用，使用关键词重叠）"]
        method = "关键词重叠"

    print(f"\n{ex['label']}")
    print(f"  问题：{ex['question']}")
    print(f"  答案（节选）：{ex['answer'][:40]}...")
    if client:
        print(f"  逆向问题：{gen_questions}")
    print(f"  相关性分数（{method}）：{score:.4f}")

print()

---
# Chapter 3 · MRR（Mean Reciprocal Rank）

本节复用 Ranking Metrics 中的 MRR 概念，对 4 条模拟检索结果计算 RR / MRR@k。

**Reciprocal Rank** = 1 / 第一个相关文档的排名位置

**MRR** = 所有查询 Reciprocal Rank 的平均值

4 条模拟检索结果，每条只有一个「正确答案」文档 ID。

## MRR@k 计算流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    RET[retrieved 列表] --> SCAN[扫描 rank 1 到 k]
    SCAN --> HIT{命中 relevant_id?}
    HIT -->|是| RR[RR = 1/rank]
    HIT -->|否| Z[RR = 0]
    RR --> AVG[MRR = 各 query RR 均值]
    Z --> AVG
```

## 排名 → RR 对照

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    R1[第1名命中] --> RR1[RR = 1.0]
    R2[第2名命中] --> RR2[RR = 0.5]
    R3[第3名命中] --> RR3[RR = 0.33]
    R0[top-k 未命中] --> RR0[RR = 0]
```

MRR@3 只扫描前 3 名；MRR@5 只扫描前 5 名。

---
# Part 3 · MRR 函数与数据（代码实战）

## 代码实现映射

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    DATA[MRR_DATA] --> RR[reciprocal_rank]
    RR --> CM[compute_mrr]
    CM --> OUT[MRR / MRR@3 / MRR@5]
```

In [ ]:
MRR_DATA = [
    {
        "query": "什么是向量数据库？",
        "retrieved": ["doc3", "doc7", "doc2", "doc5", "doc1"],
        "relevant_id": "doc2",
    },
    {
        "query": "如何评估 RAG 系统？",
        "retrieved": ["doc8", "doc1", "doc4", "doc6", "doc3"],
        "relevant_id": "doc8",
    },
    {
        "query": "什么是 HyDE？",
        "retrieved": ["doc1", "doc3", "doc4", "doc6", "doc2"],
        "relevant_id": "doc6",
    },
    {
        "query": "BM25 是什么算法？",
        "retrieved": ["doc5", "doc2", "doc8", "doc3", "doc7"],
        "relevant_id": "doc7",
    },
]


def reciprocal_rank(retrieved: list, relevant_id: str, k: int = None) -> float:
    """计算单条查询的倒数排名（Reciprocal Rank）"""
    search_list = retrieved[:k] if k else retrieved
    for rank, doc_id in enumerate(search_list, start=1):
        if doc_id == relevant_id:
            return 1.0 / rank
    return 0.0


def compute_mrr(data: list, k: int = None) -> float:
    """计算 MRR 或 MRR@k"""
    rr_scores = [reciprocal_rank(item["retrieved"], item["relevant_id"], k) for item in data]
    return sum(rr_scores) / len(rr_scores)

### 运行 MRR 评估

In [ ]:
print("=" * 60)
print("第二节：MRR（Mean Reciprocal Rank）平均倒数排名")
print("=" * 60)
print(f"\n{'查询':<20} {'正确答案':>8} {'排名':>6} {'RR':>8} {'RR@3':>8} {'RR@5':>8}")
print("-" * 65)

for item in MRR_DATA:
    rr = reciprocal_rank(item["retrieved"], item["relevant_id"])
    rr_at_3 = reciprocal_rank(item["retrieved"], item["relevant_id"], k=3)
    rr_at_5 = reciprocal_rank(item["retrieved"], item["relevant_id"], k=5)

    try:
        rank = item["retrieved"].index(item["relevant_id"]) + 1
        rank_str = str(rank)
    except ValueError:
        rank_str = "未找到"

    print(
        f"{item['query'][:18]:<20} {item['relevant_id']:>8} "
        f"{rank_str:>6} {rr:>8.4f} {rr_at_3:>8.4f} {rr_at_5:>8.4f}"
    )

mrr_all = compute_mrr(MRR_DATA)
mrr_at_3 = compute_mrr(MRR_DATA, k=3)
mrr_at_5 = compute_mrr(MRR_DATA, k=5)

print("-" * 65)
print(f"{'平均值 (MRR)':<20} {'':>8} {'':>6} {mrr_all:>8.4f} {mrr_at_3:>8.4f} {mrr_at_5:>8.4f}")

print("\n💡 解读：")
print(f"  MRR     = {mrr_all:.4f}  （不限 k，考虑所有结果）")
print(f"  MRR@3   = {mrr_at_3:.4f}  （只关注前 3 个结果）")
print(f"  MRR@5   = {mrr_at_5:.4f}  （只关注前 5 个结果）")
print("  MRR@k 越小，说明更多正确答案排在 top-k 之外，需优化检索器。")
print("=" * 60)